# EDA — Gestante de Alto Risco
**Análise exploratória completa da coorte para paper Nature Reports**
Fonte: `data/gestante_quadro_clinico.parquet` + `gestante.duckdb` (enriquecido)

In [ ]:
from __future__ import annotations
import warnings
from datetime import datetime
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
from scipy import stats as scipy_stats

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 60)
pd.set_option("display.float_format", "{:.2f}".format)

ROOT     = Path("..").resolve()
PARQUET  = ROOT / "data" / "gestante_quadro_clinico.parquet"
OUT_DIR  = ROOT / "out" / f"eda_{datetime.now().strftime('%Y%m%d_%H%M%S')}"
PLOT_DIR = OUT_DIR / "plots"
OUT_DIR.mkdir(parents=True, exist_ok=True)
PLOT_DIR.mkdir(parents=True, exist_ok=True)

## 0a. Validação dos Padrões Regex Obstétricos

In [ ]:
import subprocess, sys
_val = subprocess.run(
    [sys.executable, str(ROOT / "src" / "validar_regex_obstetrico.py"), "--somente-sintetico"],
    capture_output=True, text=True
)
print(_val.stdout[-2000:])
if "FALHAS" in _val.stdout and "0 ❌" not in _val.stdout:
    raise RuntimeError("Padrões regex obstétricos falharam — corrija antes de prosseguir.")

## 0b. Enriquecimento Obstétrico — base completa (regex + LLM → DuckDB)

In [ ]:
import sys as _sys
import duckdb
_sys.path.insert(0, str(ROOT / "src"))
from build_obstetric_features import enrich_obstetric

_DUCKDB        = ROOT / "data" / "gestante.duckdb"
_PARQUET_ENRIQ = ROOT / "data" / "gestante_quadro_clinico_enriquecido.parquet"

# Lê base completa direto do DuckDB
_con = duckdb.connect(str(_DUCKDB))
df_raw = _con.execute("SELECT * FROM gestante WHERE is_recem_nascido = 0").df()
_con.close()
df_raw.columns = df_raw.columns.str.lower()
n_raw = len(df_raw)
print(f"Gestantes brutas (DuckDB): {n_raw:,}")

# Enriquece toda a base: regex (78k) + LLM overlay (~17k)
df = enrich_obstetric(df_raw, llm_parquet=str(_DUCKDB))

# Aplica filtro de contexto — registros que não são gestações da paciente
n_antes_filtro = len(df)
n_neonatal  = int(df.get("flag_neonatal", pd.Series([0])).sum())
n_ectopica  = int(df.get("flag_ectopica", pd.Series([0])).sum())
n_puerperio = int(df.get("flag_puerperio", pd.Series([0])).sum())
df = df[(df.flag_neonatal == 0)].copy()
n_final = len(df)
print(f"Após filtro de flags: {n_final:,} registros "
      f"(excluídos: neonatal={n_neonatal}, ectópica={n_ectopica}, puerpério={n_puerperio})")

# Salva de volta no DuckDB como tabela persistente
_con = duckdb.connect(str(_DUCKDB))
_con.execute("DROP TABLE IF EXISTS gestante_enriquecido")
_con.execute("CREATE TABLE gestante_enriquecido AS SELECT * FROM df")
n_enriq = _con.execute("SELECT COUNT(*) FROM gestante_enriquecido").fetchone()[0]
_con.close()
print(f"DuckDB — gestante_enriquecido: {n_enriq:,} registros")

df.to_parquet(_PARQUET_ENRIQ, index=False)
print(f"Parquet salvo: {_PARQUET_ENRIQ}")

# Prepara colunas derivadas
for c in ["dt_atendimento","dt_alta","dt_nascimento","dt_parto"]:
    if c in df.columns:
        df[c] = pd.to_datetime(df[c], errors="coerce")

df["ano"]           = df["dt_atendimento"].dt.year
df["mes"]           = df["dt_atendimento"].dt.month
df["y_retorno_30d"] = (df["sn_retorno_30d"].fillna("N") == "S").astype(int)
df["y_retorno_90d"] = (df["sn_retorno_90d"].fillna("N") == "S").astype(int)
df["y_obito"]       = (df["sn_obito"].fillna("N") == "S").astype(int)
df["y_desfecho"]    = ((df["y_obito"]==1) |
                       (pd.to_numeric(df["qtd_obito_neonatal"],errors="coerce").fillna(0)>0)).astype(int)
df["idade_anos"]    = pd.to_numeric(df.get("idade_anos"), errors="coerce")

# IG: usa enriquecida (IG_obs) com fallback para estruturada
df["semanas_gest"]  = pd.to_numeric(df.get("IG_obs"), errors="coerce")
_struct_ig          = pd.to_numeric(df.get("semanas_gestacao"), errors="coerce")
df["semanas_gest"]  = df["semanas_gest"].fillna(_struct_ig)

# G_obs, A_obs, C_obs — vindos do enrich_obstetric
for col in ["G_obs","A_obs","C_obs","N_obs","IG_obs"]:
    if col in df.columns:
        df[col] = pd.to_numeric(df[col], errors="coerce")

def seg_cid(c):
    if pd.isna(c): return "Sem CID"
    c = str(c).upper()
    if c[:3] in [f"O{i:02d}" for i in range(10,17)]: return "Hipertensivos"
    if c[:3] == "O24": return "Diabetes Gest."
    if c[:3] in ["O60","O42","O47"]: return "Parto Prematuro"
    if c[0]=="O" and c[1] in "234": return "Hemorragia"
    if c[:3]=="O98": return "Infecção"
    if c[:3]=="O99": return "Comorbidades"
    if c[0]=="P": return "Risco Neonatal"
    if c[0]=="O": return "Outros Obstétricos"
    if c[:2]=="Z3": return "Pré-natal"
    return "Outros"

df["segmento"] = df["cd_cid"].apply(seg_cid)

print(f"\nShape final: {df.shape}")
print(f"Período: {df['dt_atendimento'].min().date()} → {df['dt_atendimento'].max().date()}")
print(f"Pacientes únicos: {df['cd_paciente'].nunique():,}")

## 0c. Fluxo CONSORT — Inclusão e Exclusão da Coorte

In [ ]:
fig_c, ax_c = plt.subplots(figsize=(8, 10))
ax_c.axis("off")

boxes = [
    (0.50, 0.95, f"Atendimentos totais no sistema MV\nn = {n_raw + n_neonatal:,}", "#4e79a7"),
    (0.50, 0.78, f"Excluídos: registros de Recém-Nascidos\nn = {n_neonatal:,}", "#f28e2b"),
    (0.50, 0.63, f"Gestantes identificadas (is_recem_nascido = 0)\nn = {n_raw:,}", "#4e79a7"),
    (0.50, 0.46, f"Excluídos por contexto clínico\nneonatal={n_neonatal} | ectópica={n_ectopica} | puerpério={n_puerperio}", "#f28e2b"),
    (0.50, 0.30, f"Coorte final para análise\nn = {n_final:,}", "#59a14f"),
    (0.50, 0.13, f"Com enriquecimento obstétrico\n(regex + LLM overlay)\nn_LLM ≈ 17.000 (~{17000/n_final*100:.0f}% da coorte)", "#76b7b2"),
]

for x, y, txt, cor in boxes:
    ax_c.add_patch(plt.Rectangle((x-0.32, y-0.07), 0.64, 0.10,
                                  facecolor=cor, edgecolor="white", alpha=0.85,
                                  transform=ax_c.transAxes, clip_on=False))
    ax_c.text(x, y-0.02, txt, ha="center", va="center", fontsize=9,
              color="white", fontweight="bold", transform=ax_c.transAxes)

# Setas
for y_start, y_end in [(0.88, 0.85), (0.71, 0.68), (0.53, 0.50), (0.37, 0.34), (0.20, 0.17)]:
    ax_c.annotate("", xy=(0.5, y_end), xytext=(0.5, y_start),
                  xycoords="axes fraction", textcoords="axes fraction",
                  arrowprops=dict(arrowstyle="->", color="gray", lw=1.5))

ax_c.set_title("Fluxo CONSORT — Seleção da Coorte de Gestantes de Alto Risco",
               fontsize=12, pad=15)
plt.tight_layout()
plt.savefig(PLOT_DIR/"00_consort_flow.png", dpi=150, bbox_inches="tight")
plt.show(); plt.close()
print("  Gerado: plots/00_consort_flow.png")

## 1. Qualidade dos Dados — Cobertura dos Campos-Chave

In [ ]:
# Campos estruturados (Oracle) + campos enriquecidos via texto
cols_interesse = {
    # Oracle estruturado
    "cd_cid":                    "CID principal",
    "dt_nascimento":             "Data nascimento paciente",
    "semanas_gestacao":          "IG estruturada (ADMISSAO_CO)",
    "qt_gestacao":               "Nº gestações estruturado",
    "qt_paridade":               "Paridade estruturada",
    "qt_cesaria":                "Cesáreas estruturadas",
    "sn_prematuro":              "Flag prematuro (TISS)",
    "nr_consultas_pre_natal":    "Nº consultas pré-natal",
    "vl_escore_triagem":         "Escore Manchester",
    "cd_cor_manchester":         "Cor Manchester",
    "media_peso_rn_g":           "Peso RN (gramas)",
    "min_apgar_5min":            "Apgar 5min",
    # Enriquecido via texto (regex + LLM)
    "G_obs":                     "G — gestações (enriquecido)",
    "A_obs":                     "A — abortos (enriquecido)",
    "C_obs":                     "C — cesáreas (enriquecido)",
    "IG_obs":                    "IG semanas (enriquecido)",
    "N_obs":                     "N — filhos vivos (enriquecido)",
    "tipo_gestacao":             "Tipo gestação (regex)",
    "perfil_historico":          "Perfil obstétrico (derivado)",
    # Texto clínico
    "ds_queixa_principal":       "Queixa principal (texto)",
    "txt_evolucao_medica":       "Evolução médica (texto)",
    "notacao_obstetrica":        "Notação obstétrica (texto)",
}

qual_rows = []
for col, label in cols_interesse.items():
    n_ok  = df[col].notna().sum() if col in df.columns else 0
    pct   = round(df[col].notna().mean()*100, 1) if col in df.columns else 0
    tipo  = "enriquecido" if col in ["G_obs","A_obs","C_obs","IG_obs","N_obs","tipo_gestacao","perfil_historico"] else "estruturado/texto"
    qual_rows.append({"campo": col, "label": label, "n": n_ok, "pct": pct, "tipo": tipo})

qual = pd.DataFrame(qual_rows).sort_values("pct", ascending=False)
print("\nCobertura dos campos-chave:"); print(qual[["campo","pct","tipo"]].to_string(index=False))
qual.to_csv(OUT_DIR/"qualidade_campos.csv", index=False)

fig, ax = plt.subplots(figsize=(12, 8))
cores = []
for _, row in qual.iterrows():
    if row["tipo"] == "enriquecido":
        cores.append("#76b7b2" if row["pct"] >= 50 else "#f28e2b")
    else:
        cores.append("#2ca02c" if row["pct"] >= 70 else "#ff7f0e" if row["pct"] >= 30 else "#d62728")
ax.barh(qual["label"][::-1], qual["pct"][::-1], color=cores[::-1], alpha=0.85)
ax.axvline(70, color="green",  ls="--", lw=0.8, alpha=0.8, label="70%")
ax.axvline(30, color="orange", ls="--", lw=0.8, alpha=0.8, label="30%")
ax.set_xlabel("% preenchido"); ax.set_title("Cobertura dos Campos-Chave\n(azul=enriquecido texto | verde/laranja/vermelho=estruturado)")
ax.legend(fontsize=9); ax.grid(axis="x", alpha=0.3); plt.tight_layout()
plt.savefig(PLOT_DIR/"01_cobertura.png", dpi=150, bbox_inches="tight"); plt.show(); plt.close()

## 2. Tabela 1 — Caracterização da Coorte (TRIPOD-AI)

In [ ]:
def pct(cond):  return f"{cond.sum():,} ({cond.mean()*100:.1f}%)"
def med_iqr(s): q1,q3=s.quantile([.25,.75]); return f"{s.median():.1f} [{q1:.1f}–{q3:.1f}]"
def chi2_p(cond):
    """P-valor chi² retorno_30d vs condição binária."""
    try:
        ct = pd.crosstab(cond.astype(int), df["y_retorno_30d"])
        _, p, _, _ = scipy_stats.chi2_contingency(ct)
        return f"{p:.4f}"
    except Exception:
        return "—"
def mw_p(s):
    """P-valor Mann-Whitney retorno_30d vs variável contínua."""
    try:
        g1 = s[df["y_retorno_30d"]==1].dropna()
        g0 = s[df["y_retorno_30d"]==0].dropna()
        _, p = scipy_stats.mannwhitneyu(g1, g0, alternative="two-sided")
        return f"{p:.4f}"
    except Exception:
        return "—"

rows = {
    "N atendimentos":              (f"{len(df):,}", "—"),
    "N pacientes únicos":          (f"{df['cd_paciente'].nunique():,}", "—"),
    "Atend./paciente — med [IQR]": (med_iqr(df.groupby("cd_paciente").size()), "—"),
    "── Setor ──":                 ("", ""),
    "  Ambulatório (A)":           (pct(df["tp_atendimento"]=="A"), chi2_p(df["tp_atendimento"]=="A")),
    "  Urgência/URI (U)":          (pct(df["tp_atendimento"]=="U"), chi2_p(df["tp_atendimento"]=="U")),
    "  Internação (I)":            (pct(df["tp_atendimento"]=="I"), chi2_p(df["tp_atendimento"]=="I")),
    "── Demográfico ──":           ("", ""),
    "  Idade — med [IQR]":         (med_iqr(df["idade_anos"].dropna()), mw_p(df["idade_anos"])),
    "  < 10 anos (caso extremo)":    (pct(df["idade_anos"]<10),             chi2_p(df["idade_anos"]<10)),
    "  10–14 anos (abuso/prec.)":   (pct(df["idade_anos"].between(10,14)), chi2_p(df["idade_anos"].between(10,14))),
    "  15–19 anos (adolescente)":   (pct(df["idade_anos"].between(15,19)), chi2_p(df["idade_anos"].between(15,19))),
    "  20–24 anos":                 (pct(df["idade_anos"].between(20,24)), chi2_p(df["idade_anos"].between(20,24))),
    "  25–29 anos":                 (pct(df["idade_anos"].between(25,29)), chi2_p(df["idade_anos"].between(25,29))),
    "  30–34 anos":                 (pct(df["idade_anos"].between(30,34)), chi2_p(df["idade_anos"].between(30,34))),
    "  35–39 anos (id. avançada)": (pct(df["idade_anos"].between(35,39)), chi2_p(df["idade_anos"].between(35,39))),
    "  40–44 anos":                 (pct(df["idade_anos"].between(40,44)), chi2_p(df["idade_anos"].between(40,44))),
    "  ≥ 45 anos":                  (pct(df["idade_anos"]>=45),            chi2_p(df["idade_anos"]>=45)),
    "── CID principal ──":         ("", ""),
    **{f"  {s}": (pct(df["segmento"]==s), chi2_p(df["segmento"]==s))
       for s in df["segmento"].value_counts().index[:7]},
    "── Obstétrico (enriquecido) ──": ("", ""),
    "  IG semanas — med [IQR]":    (med_iqr(df["semanas_gest"].dropna()), mw_p(df["semanas_gest"])),
    "  Prematuro (<37s)":          (pct(df["semanas_gest"]<37), chi2_p(df["semanas_gest"]<37)),
    "  Muito prematuro (<32s)":    (pct(df["semanas_gest"]<32), chi2_p(df["semanas_gest"]<32)),
    "  Primigesta (G=1)":          (pct(df.get("G_obs",pd.Series())==1) if "G_obs" in df.columns else ("—","—"),
                                    chi2_p(df["G_obs"]==1) if "G_obs" in df.columns else "—"),
    "  G — med [IQR]":             (med_iqr(df["G_obs"].dropna()) if "G_obs" in df.columns else "—",
                                    mw_p(df["G_obs"]) if "G_obs" in df.columns else "—"),
    "  A ≥ 1 (aborto prévio)":     (pct(df["A_obs"]>=1) if "A_obs" in df.columns else "—",
                                    chi2_p(df["A_obs"]>=1) if "A_obs" in df.columns else "—"),
    "  C ≥ 1 (cesárea prévia)":    (pct(df["C_obs"]>=1) if "C_obs" in df.columns else "—",
                                    chi2_p(df["C_obs"]>=1) if "C_obs" in df.columns else "—"),
    "── Triagem Manchester ──":    ("", ""),
    "  Com triagem registrada":    (pct(df["vl_escore_triagem"].notna()), chi2_p(df["vl_escore_triagem"].notna())),
    "  Escore — med [IQR]":        (med_iqr(df["vl_escore_triagem"].dropna()), mw_p(df["vl_escore_triagem"])),
    "  Vermelho/Laranja (urgente)": (pct(df.get("cd_cor_manchester",pd.Series(dtype=float)).fillna(5)<=2)
                                     if "cd_cor_manchester" in df.columns else ("—","—"),
                                    chi2_p(df["cd_cor_manchester"].fillna(5)<=2) if "cd_cor_manchester" in df.columns else "—"),
    "── Desfechos ──":             ("", ""),
    "  Retorno 30d":               (pct(df["y_retorno_30d"]==1), "—"),
    "  Retorno 90d":               (pct(df["y_retorno_90d"]==1), "—"),
    "  Óbito materno":             (pct(df["y_obito"]==1), "—"),
    "  Óbito neonatal":            (pct(pd.to_numeric(df["qtd_obito_neonatal"],errors="coerce").fillna(0)>0), "—"),
    "── Documentação clínica ──":  ("", ""),
    "  Com queixa principal":      (pct(df["ds_queixa_principal"].notna()), chi2_p(df["ds_queixa_principal"].notna())),
    "  Com evolução médica":       (pct(df["txt_evolucao_medica"].notna()), chi2_p(df["txt_evolucao_medica"].notna())),
}

tab1_rows = []
for k, v in rows.items():
    if isinstance(v, tuple):
        tab1_rows.append({"Característica": k, "Valor": v[0], "p (retorno 30d)": v[1]})
    else:
        tab1_rows.append({"Característica": k, "Valor": v, "p (retorno 30d)": "—"})

tab1 = pd.DataFrame(tab1_rows)
print("\nTabela 1 — Coorte de Gestantes de Alto Risco:")
print(tab1.to_string(index=False))
tab1.to_csv(OUT_DIR/"tabela1_coorte.csv", index=False)

## 3. Distribuição Temporal e Tendência de Retorno

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(16, 10))

# Volume por ano
vol = df.groupby("ano").size()
vol.plot(ax=axes[0,0], kind="bar", color="steelblue", edgecolor="white")
axes[0,0].set_title("Atendimentos por Ano"); axes[0,0].set_ylabel("N")
axes[0,0].grid(axis="y", alpha=0.3)
for i, v in enumerate(vol):
    axes[0,0].text(i, v + 50, f"{v:,}", ha="center", fontsize=7)

# Taxa retorno 30d por ano com IC 95% bootstrap
ret_ano = df.groupby("ano")["y_retorno_30d"].agg(["mean","count"]).reset_index()
ret_ano["pct"] = ret_ano["mean"] * 100
ret_ano["se"]  = np.sqrt(ret_ano["mean"] * (1 - ret_ano["mean"]) / ret_ano["count"]) * 100 * 1.96
axes[0,1].plot(ret_ano["ano"], ret_ano["pct"], "o-", color="tomato", lw=2, markersize=6)
axes[0,1].fill_between(ret_ano["ano"],
                        ret_ano["pct"] - ret_ano["se"],
                        ret_ano["pct"] + ret_ano["se"],
                        alpha=0.2, color="tomato", label="IC 95%")
axes[0,1].set_title("Taxa de Retorno 30d por Ano (% ± IC95%)"); axes[0,1].set_ylabel("%")
axes[0,1].legend(fontsize=9); axes[0,1].grid(alpha=0.3)

# Retorno por mês (sazonalidade)
ret_mes = df.groupby("mes")["y_retorno_30d"].mean() * 100
meses = ["Jan","Fev","Mar","Abr","Mai","Jun","Jul","Ago","Set","Out","Nov","Dez"]
axes[1,0].bar(range(1,13), ret_mes.reindex(range(1,13)).fillna(0),
              color="steelblue", edgecolor="white", alpha=0.85)
axes[1,0].set_xticks(range(1,13)); axes[1,0].set_xticklabels(meses, rotation=45, fontsize=8)
axes[1,0].set_title("Taxa de Retorno 30d por Mês (sazonalidade)"); axes[1,0].set_ylabel("%")
axes[1,0].grid(axis="y", alpha=0.3)

# Hora de admissão (padrão diário)
ret_hora = df.groupby(df["dt_atendimento"].dt.hour)["y_retorno_30d"].agg(["mean","count"])
axes[1,1].bar(ret_hora.index, ret_hora["mean"]*100, color="steelblue", edgecolor="white", alpha=0.75)
axes[1,1].set_xlabel("Hora de admissão"); axes[1,1].set_ylabel("Taxa retorno 30d (%)")
axes[1,1].set_title("Retorno 30d por Hora de Admissão"); axes[1,1].grid(axis="y", alpha=0.3)

plt.suptitle("Análise Temporal — Coorte de Gestantes", fontsize=13, y=1.01)
plt.tight_layout()
plt.savefig(PLOT_DIR/"02_temporal.png", dpi=150, bbox_inches="tight"); plt.show(); plt.close()

## 4. Perfil por Segmento CID

In [ ]:
seg = df.groupby("segmento").agg(
    n=("cd_atendimento","count"),
    retorno_30d=("y_retorno_30d","mean"),
    retorno_90d=("y_retorno_90d","mean"),
    obito=("y_obito","mean"),
    ig=("semanas_gest","median"),
    idade=("idade_anos","median"),
).reset_index()
for c in ["retorno_30d","retorno_90d","obito"]:
    seg[c] = (seg[c]*100).round(2)
seg = seg.sort_values("n", ascending=False)
print("\nPerfil por Segmento CID:"); print(seg.to_string(index=False))
seg.to_csv(OUT_DIR/"perfil_segmento.csv", index=False)

fig, axes = plt.subplots(1, 3, figsize=(18, 6))
top = seg.head(9)

axes[0].barh(top["segmento"][::-1], top["n"][::-1], color="steelblue", alpha=0.85)
axes[0].set_title("Volume por Segmento CID"); axes[0].grid(axis="x", alpha=0.3)
for i, (_, row) in enumerate(top[::-1].iterrows()):
    axes[0].text(row["n"] + 50, i, f"{row['n']:,}", va="center", fontsize=8)

cores_ret = ["#d62728" if v>25 else "#ff7f0e" if v>15 else "#2ca02c" for v in top["retorno_30d"]]
axes[1].barh(top["segmento"][::-1], top["retorno_30d"][::-1], color=cores_ret[::-1], alpha=0.85)
axes[1].axvline(df["y_retorno_30d"].mean()*100, color="k", ls="--", lw=1.2, label="Média global")
axes[1].set_title("Retorno 30d por Segmento (%)"); axes[1].legend(fontsize=9); axes[1].grid(axis="x", alpha=0.3)

cores_90 = ["#d62728" if v>35 else "#ff7f0e" if v>20 else "#2ca02c" for v in top["retorno_90d"]]
axes[2].barh(top["segmento"][::-1], top["retorno_90d"][::-1], color=cores_90[::-1], alpha=0.85)
axes[2].axvline(df["y_retorno_90d"].mean()*100, color="k", ls="--", lw=1.2, label="Média global")
axes[2].set_title("Retorno 90d por Segmento (%)"); axes[2].legend(fontsize=9); axes[2].grid(axis="x", alpha=0.3)

plt.suptitle("Perfil por Segmento CID Principal", fontsize=13, y=1.01)
plt.tight_layout(); plt.savefig(PLOT_DIR/"03_segmento.png", dpi=150, bbox_inches="tight"); plt.show(); plt.close()

## 5. Distribuição Demográfica e Obstétrica

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(18, 10))

# Idade
df["idade_anos"].dropna().hist(ax=axes[0,0], bins=30, color="steelblue", edgecolor="white")
axes[0,0].axvline(10, color="darkred",  ls=":",  lw=1.2, label="10a (abuso/precoce)")
axes[0,0].axvline(15, color="red",     ls="--", lw=1.2, label="15a (adolescente)")
axes[0,0].axvline(20, color="orange",  ls="--", lw=0.8, label="20a")
axes[0,0].axvline(35, color="purple",  ls="--", lw=1.2, label="35a (id. avançada)")
axes[0,0].axvline(45, color="brown",   ls=":",  lw=1.0, label="45a")
axes[0,0].set_title(f"Distribuição de Idade\nmed={df['idade_anos'].median():.1f}a")
axes[0,0].legend(fontsize=8); axes[0,0].grid(alpha=0.3)

# IG semanas (enriquecida)
df["semanas_gest"].dropna().hist(ax=axes[0,1], bins=30, color="seagreen", edgecolor="white")
axes[0,1].axvline(37, color="red",    ls="--", lw=1.2, label="37s (prematuro)")
axes[0,1].axvline(32, color="orange", ls="--", lw=1.2, label="32s (mt. prematuro)")
pct_prem = (df["semanas_gest"]<37).mean()*100
axes[0,1].set_title(f"IG semanas (enriquecida)\n{pct_prem:.1f}% prematuros")
axes[0,1].legend(fontsize=8); axes[0,1].grid(alpha=0.3)

# Retorno 30d por setor
s = df.groupby("tp_atendimento")["y_retorno_30d"].agg(["mean","count"]).reset_index()
s["mean"] *= 100
s["tp_atendimento"] = s["tp_atendimento"].map({"A":"Ambulatório","U":"Urgência","I":"Internação"}).fillna(s["tp_atendimento"])
axes[0,2].bar(s["tp_atendimento"], s["mean"], color=["steelblue","tomato","seagreen"], alpha=0.85)
axes[0,2].set_title("Retorno 30d por Setor (%)"); axes[0,2].set_ylabel("%")
axes[0,2].grid(axis="y", alpha=0.3)
for i,r in s.iterrows():
    axes[0,2].text(i, r["mean"]+0.3, f"n={int(r['count']):,}", ha="center", fontsize=8)

# G_obs — gestações
if "G_obs" in df.columns:
    g_vals = df["G_obs"].dropna().clip(1, 10)
    vc = g_vals.value_counts().sort_index()
    axes[1,0].bar(vc.index.astype(int), vc.values, color="steelblue", edgecolor="white", alpha=0.85)
    axes[1,0].set_xlabel("Número de gestações (G_obs)")
    axes[1,0].set_title(f"Distribuição de G (gestações)\nmed={g_vals.median():.0f}  n={len(g_vals):,}")
    axes[1,0].grid(axis="y", alpha=0.3)

# A_obs — abortos
if "A_obs" in df.columns:
    a_vals = df["A_obs"].dropna().clip(0, 6)
    vc_a = a_vals.value_counts().sort_index()
    axes[1,1].bar(vc_a.index.astype(int), vc_a.values, color="tomato", edgecolor="white", alpha=0.85)
    axes[1,1].set_xlabel("Número de abortos (A_obs)")
    axes[1,1].set_title(f"Distribuição de A (abortos)\n{(df['A_obs']>=1).mean()*100:.1f}% com A≥1")
    axes[1,1].grid(axis="y", alpha=0.3)

# tipo_gestacao
if "tipo_gestacao" in df.columns:
    tp = df["tipo_gestacao"].value_counts().head(8)
    axes[1,2].barh(tp.index[::-1], tp.values[::-1], color="steelblue", edgecolor="white", alpha=0.85)
    axes[1,2].set_title(f"Tipos de Gestação (regex)\nn={df['tipo_gestacao'].notna().sum():,} registros")
    axes[1,2].set_xlabel("N")
    axes[1,2].grid(axis="x", alpha=0.3)

plt.suptitle("Distribuição Demográfica e Obstétrica", fontsize=13, y=1.01)
plt.tight_layout(); plt.savefig(PLOT_DIR/"04_demografico.png", dpi=150, bbox_inches="tight"); plt.show(); plt.close()

## 6. Risco LLM — Alertas e Distribuição por Segmento

In [ ]:
try:
    _con_llm = duckdb.connect(str(_DUCKDB))
    llm_n    = _con_llm.execute("SELECT COUNT(*) FROM llm_features").fetchone()[0]
    llm_risk = _con_llm.execute("SELECT * FROM v_llm_risco_composto").df()
    llm_seg  = _con_llm.execute("SELECT * FROM v_llm_por_segmento ORDER BY total_alertas DESC").df()
    llm_alrt = _con_llm.execute("SELECT * FROM v_llm_alertas LIMIT 5").df() if \
               "v_llm_alertas" in _con_llm.execute("SHOW TABLES").df()["name"].tolist() else pd.DataFrame()
    _con_llm.close()

    print(f"\nRegistros com LLM: {llm_n:,} ({llm_n/len(df)*100:.1f}% da coorte)")
    print("\nDistribuição de risco LLM:")
    print(llm_risk.to_string(index=False))

    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    cores_r = {"CRÍTICO":"#d62728","ALTO":"#ff7f0e","MODERADO":"#ffd700","BAIXO":"#2ca02c"}
    cores_b = [cores_r.get(str(r).upper(),"gray") for r in llm_risk.get("risco_composto", llm_risk.iloc[:,0])]
    col_risco  = llm_risk.columns[0]
    col_total  = [c for c in llm_risk.columns if "total" in c.lower()][0]
    col_pct    = [c for c in llm_risk.columns if "pct" in c.lower()][0] if any("pct" in c.lower() for c in llm_risk.columns) else None

    axes[0].bar(llm_risk[col_risco], llm_risk[col_total], color=cores_b, edgecolor="white", alpha=0.85)
    axes[0].set_title(f"Risco Composto LLM (n={llm_n:,})")
    axes[0].grid(axis="y", alpha=0.3)
    if col_pct:
        for i, r in llm_risk.iterrows():
            axes[0].text(i, r[col_total]+20, f"{r[col_pct]}%", ha="center", fontsize=9)

    top_seg = llm_seg.head(7)
    col_seg   = llm_seg.columns[0]
    col_palrt = [c for c in llm_seg.columns if "pct" in c.lower()][0] if any("pct" in c.lower() for c in llm_seg.columns) else llm_seg.columns[1]
    axes[1].barh(top_seg[col_seg][::-1], top_seg[col_palrt][::-1], color="tomato", edgecolor="white", alpha=0.85)
    axes[1].set_title("% Alertas (red_flag) por Segmento"); axes[1].set_xlabel("%")
    axes[1].grid(axis="x", alpha=0.3)

    plt.tight_layout(); plt.savefig(PLOT_DIR/"05_risco_llm.png", dpi=150, bbox_inches="tight"); plt.show(); plt.close()
    llm_risk.to_csv(OUT_DIR/"llm_risco_composto.csv", index=False)
    llm_seg.to_csv(OUT_DIR/"llm_por_segmento.csv", index=False)

except Exception as e:
    print(f"LLM features não disponíveis: {e}")

## 7. EDA dos Recém-Nascidos — Segundo Público

In [ ]:
try:
    _pq_abs = str(PARQUET.resolve())
    _con_rn = duckdb.connect(str(ROOT/"data"/"gestante.duckdb"))

    df_rn = _con_rn.execute("""
        SELECT
            r.cd_atendimento          AS cd_atend_rn,
            r.cd_paciente             AS cd_paciente_rn,
            r.cd_atendimento_pai      AS cd_atend_mae,
            r.sn_obito                AS obito_rn,
            r.media_peso_rn_g         AS peso_g,
            r.min_apgar_1min          AS apgar1,
            r.min_apgar_5min          AS apgar5,
            r.qtd_rn_baixo_peso       AS baixo_peso,
            g.cd_cid                  AS cid_mae,
            g.idade_anos              AS idade_mae,
            g.semanas_gestacao        AS ig_semanas,
            g.sn_gestacao_tiss        AS gestacao_tiss,
            g.sn_cesario_tiss         AS cesario_tiss
        FROM recem_nascido_atendimento r
        LEFT JOIN gestante_enriquecido g ON g.cd_atendimento = r.cd_atendimento_pai
        WHERE r.cd_atendimento_pai IS NOT NULL
    """).df()
    _con_rn.close()

    df_rn.columns = df_rn.columns.str.lower()
    # segmento_cid é derivado em Python — não está armazenado no DuckDB
    df_rn["segmento_mae"] = df_rn["cid_mae"].apply(seg_cid)
    df_rn["obito_rn"]      = (df_rn["obito_rn"].fillna("N") == "S").astype(int)
    df_rn["peso_g"]        = pd.to_numeric(df_rn["peso_g"],       errors="coerce")
    df_rn["apgar5"]        = pd.to_numeric(df_rn["apgar5"],       errors="coerce")
    df_rn["ig_semanas"]    = pd.to_numeric(df_rn["ig_semanas"],   errors="coerce")
    df_rn["apgar_deprimido"]  = (df_rn["apgar5"] < 7).astype(float)
    df_rn["baixo_peso_flag"]  = (df_rn["peso_g"] < 2500).astype(float)
    df_rn["muito_baixo_peso"] = (df_rn["peso_g"] < 1500).astype(float)
    df_rn["prematuro"]        = (df_rn["ig_semanas"] < 37).astype(float)

    print(f"\nEDA Recém-Nascidos")
    print(f"  Total RN vinculados: {len(df_rn):,}")
    print(f"  Óbitos neonatais:    {df_rn['obito_rn'].sum():,} ({df_rn['obito_rn'].mean()*100:.2f}%)")
    print(f"  Com peso:            {df_rn['peso_g'].notna().sum():,}")
    print(f"  Apgar < 7 (5min):    {df_rn['apgar_deprimido'].sum():,.0f}")
    print(f"  Baixo peso (<2500g): {df_rn['baixo_peso_flag'].sum():,.0f}")

    fig, axes = plt.subplots(1, 3, figsize=(16, 5))

    df_rn["peso_g"].dropna().hist(ax=axes[0], bins=40, color="steelblue", edgecolor="white")
    axes[0].axvline(2500, color="red",   ls="--", lw=1.5, label="2500g")
    axes[0].axvline(1500, color="orange",ls="--", lw=1.5, label="1500g")
    axes[0].axvline(4000, color="green", ls="--", lw=1.5, label="4000g")
    axes[0].set_title("Peso ao Nascer (g)"); axes[0].legend(fontsize=8); axes[0].grid(alpha=0.3)

    seg_ob = df_rn.groupby("segmento_mae")["obito_rn"].agg(["mean","count"]).reset_index()
    seg_ob = seg_ob[seg_ob["count"] >= 10].sort_values("mean", ascending=False)
    seg_ob["mean"] *= 100
    cores_ob = ["#d62728" if v > 2 else "#ff7f0e" if v > 0.5 else "#2ca02c" for v in seg_ob["mean"]]
    axes[1].barh(seg_ob["segmento_mae"][::-1], seg_ob["mean"][::-1], color=cores_ob[::-1])
    axes[1].set_xlabel("Taxa de óbito neonatal (%)"); axes[1].set_title("Óbito Neonatal por CID Materno")
    axes[1].grid(axis="x", alpha=0.3)

    for vals, lbl, cor in [
        (df_rn[df_rn["ig_semanas"]>=37]["apgar5"].dropna(), "≥37s (termo)", "steelblue"),
        (df_rn[(df_rn["ig_semanas"]>=32)&(df_rn["ig_semanas"]<37)]["apgar5"].dropna(), "32–36s", "orange"),
        (df_rn[df_rn["ig_semanas"]<32]["apgar5"].dropna(), "<32s (mt. prem.)", "red"),
    ]:
        if len(vals):
            axes[2].hist(vals, bins=10, alpha=0.6, label=f"{lbl} (n={len(vals)})",
                         density=True, color=cor, edgecolor="white")
    axes[2].axvline(7, color="black", ls="--", lw=1, label="Apgar<7")
    axes[2].set_title("Apgar 5min por Prematuridade"); axes[2].legend(fontsize=8); axes[2].grid(alpha=0.3)

    plt.suptitle("EDA Recém-Nascidos — Cruzamento com Perfil Materno", y=1.01, fontsize=12)
    plt.tight_layout(); plt.savefig(PLOT_DIR/"06_eda_recem_nascidos.png", dpi=150, bbox_inches="tight"); plt.show(); plt.close()

    if df_rn["ig_semanas"].notna().sum() > 50:
        df_rn["ig_grupo"] = pd.cut(df_rn["ig_semanas"], bins=[0,28,32,34,37,40,50],
                                    labels=["<28s","28-31s","32-33s","34-36s","37-40s",">40s"], right=False)
        ig_peso = df_rn.groupby("ig_grupo", observed=True).agg(
            n=("peso_g","count"), peso_medio=("peso_g","mean"), pct_obito=("obito_rn","mean")
        ).reset_index()
        ig_peso["peso_medio"] = ig_peso["peso_medio"].round(0)
        ig_peso["pct_obito"]  = (ig_peso["pct_obito"]*100).round(2)
        print("\nPeso médio e óbito por IG:"); print(ig_peso.to_string(index=False))

    df_rn.to_csv(OUT_DIR/"eda_recem_nascidos.csv", index=False)
    seg_ob.to_csv(OUT_DIR/"obito_neonatal_por_segmento_materno.csv", index=False)

except Exception as e:
    print(f"EDA RN não disponível: {e}")
    import traceback; traceback.print_exc()

## 8. Perfil Obstétrico Enriquecido — G / A / C / IG / tipo_gestacao

In [ ]:
print("\n" + "="*70)
print("PERFIL OBSTÉTRICO ENRIQUECIDO (regex + LLM)")
print("="*70)

# Cobertura por fonte
for col, lbl in [("G_obs","G (gestações)"), ("A_obs","A (abortos)"), ("C_obs","C (cesáreas)"),
                  ("IG_obs","IG semanas"), ("N_obs","N (nascidos vivos)"), ("tipo_gestacao","Tipo gestação")]:
    if col in df.columns:
        n_ok  = df[col].notna().sum()
        n_struct = df.get({"G_obs":"qt_gestacao","A_obs":"qt_aborto","C_obs":"qt_cesaria",
                            "IG_obs":"semanas_gestacao"}.get(col, col), pd.Series()).notna().sum()
        print(f"  {lbl:<25}: enriquecido={n_ok:>6,} ({n_ok/len(df)*100:.1f}%)  "
              f"estruturado={n_struct:>6,} ({n_struct/len(df)*100:.1f}%)")

# Plot perfis
if all(c in df.columns for c in ["G_obs","A_obs","IG_obs"]):
    fig, axes = plt.subplots(2, 3, figsize=(18, 10))

    # G × retorno_30d
    g_ret = df.groupby(df["G_obs"].clip(1,8).dropna().astype(int))["y_retorno_30d"].mean() * 100
    axes[0,0].bar(g_ret.index, g_ret.values, color="steelblue", edgecolor="white", alpha=0.85)
    axes[0,0].axhline(df["y_retorno_30d"].mean()*100, color="black", ls="--", lw=1, label="Média global")
    axes[0,0].set_xlabel("G (gestações)"); axes[0,0].set_ylabel("Retorno 30d (%)")
    axes[0,0].set_title("Retorno 30d por Nº de Gestações (G)"); axes[0,0].legend(fontsize=8); axes[0,0].grid(alpha=0.3)

    # A × retorno_30d
    a_ret = df.groupby(df["A_obs"].clip(0,5).dropna().astype(int))["y_retorno_30d"].mean() * 100
    axes[0,1].bar(a_ret.index, a_ret.values, color="tomato", edgecolor="white", alpha=0.85)
    axes[0,1].axhline(df["y_retorno_30d"].mean()*100, color="black", ls="--", lw=1)
    axes[0,1].set_xlabel("A (abortos)"); axes[0,1].set_ylabel("Retorno 30d (%)")
    axes[0,1].set_title("Retorno 30d por Nº de Abortos (A)"); axes[0,1].grid(alpha=0.3)

    # IG × retorno_30d
    ig_bins = [0,28,32,34,37,40,45]
    ig_lbl  = ["<28s","28-31s","32-33s","34-36s","37-40s",">40s"]
    df["ig_grupo"] = pd.cut(df["semanas_gest"], bins=ig_bins, labels=ig_lbl, right=False)
    ig_ret = df.groupby("ig_grupo", observed=True)["y_retorno_30d"].agg(["mean","count"]).reset_index()
    ig_ret["mean"] *= 100
    axes[0,2].bar(ig_ret["ig_grupo"].astype(str), ig_ret["mean"],
                  color="seagreen", edgecolor="white", alpha=0.85)
    axes[0,2].axhline(df["y_retorno_30d"].mean()*100, color="black", ls="--", lw=1)
    axes[0,2].set_xlabel("IG semanas (grupos)"); axes[0,2].set_ylabel("Retorno 30d (%)")
    axes[0,2].set_title("Retorno 30d por IG"); axes[0,2].grid(axis="y", alpha=0.3)
    for i, r in ig_ret.iterrows():
        axes[0,2].text(i, r["mean"]+0.3, f"n={int(r['count']):,}", ha="center", fontsize=7)

    # Tipo gestação × volume
    if "tipo_gestacao" in df.columns:
        tp_vc = df["tipo_gestacao"].value_counts().head(10)
        axes[1,0].barh(tp_vc.index[::-1], tp_vc.values[::-1], color="steelblue", edgecolor="white", alpha=0.85)
        axes[1,0].set_title(f"Distribuição tipo_gestacao\n(n={df['tipo_gestacao'].notna().sum():,})")
        axes[1,0].set_xlabel("N"); axes[1,0].grid(axis="x", alpha=0.3)

    # perfil_historico
    if "perfil_historico" in df.columns:
        ph_vc = df["perfil_historico"].value_counts().head(8)
        axes[1,1].barh(ph_vc.index[::-1], ph_vc.values[::-1], color="tomato", edgecolor="white", alpha=0.85)
        axes[1,1].set_title(f"Perfil Histórico Obstétrico\n(n={df['perfil_historico'].notna().sum():,})")
        axes[1,1].set_xlabel("N"); axes[1,1].grid(axis="x", alpha=0.3)

    # G × A scatter (amostra)
    mask = df["G_obs"].notna() & df["A_obs"].notna()
    smp = df[mask].sample(min(3000, mask.sum()), random_state=42)
    axes[1,2].scatter(smp["G_obs"].clip(1,10), smp["A_obs"].clip(0,6),
                      c=smp["y_retorno_30d"], cmap="RdYlGn_r", alpha=0.3, s=10)
    axes[1,2].set_xlabel("G (gestações)"); axes[1,2].set_ylabel("A (abortos)")
    axes[1,2].set_title("G × A — retorno_30d\n(verde=sem retorno, vermelho=retorno)")
    axes[1,2].grid(alpha=0.3)

    plt.suptitle("Perfil Obstétrico Enriquecido (regex + LLM)", fontsize=13, y=1.01)
    plt.tight_layout(); plt.savefig(PLOT_DIR/"07_perfil_obstetrico.png", dpi=150, bbox_inches="tight"); plt.show(); plt.close()
    print("  Gerado: plots/07_perfil_obstetrico.png")

## 9. Comparação Estruturado vs Enriquecido — Ganho do Texto

In [ ]:
print("\n" + "="*70)
print("COMPARAÇÃO: CAMPOS ESTRUTURADOS (Oracle) vs ENRIQUECIDOS (texto)")
print("="*70)

comparacoes = [
    ("qt_gestacao",     "G_obs",  "Nº Gestações (G)"),
    ("qt_aborto",       "A_obs",  "Nº Abortos (A)"),
    ("qt_cesaria",      "C_obs",  "Nº Cesáreas (C)"),
    ("semanas_gestacao","IG_obs", "IG Semanas"),
]

rows_comp = []
for col_struct, col_enriq, lbl in comparacoes:
    n_struct = df[col_struct].notna().sum() if col_struct in df.columns else 0
    n_enriq  = df[col_enriq].notna().sum()  if col_enriq  in df.columns else 0
    ganho    = n_enriq - n_struct
    ganho_pct= ganho / len(df) * 100
    rows_comp.append({
        "Campo": lbl,
        "Estruturado (n)": n_struct,
        "Estruturado (%)": round(n_struct/len(df)*100, 1),
        "Enriquecido (n)": n_enriq,
        "Enriquecido (%)": round(n_enriq/len(df)*100, 1),
        "Ganho (n)":       ganho,
        "Ganho (%)":       round(ganho_pct, 1),
    })
    print(f"  {lbl:<20}: estrut={n_struct:>6,} ({n_struct/len(df)*100:.1f}%)  "
          f"enriq={n_enriq:>6,} ({n_enriq/len(df)*100:.1f}%)  "
          f"ganho=+{ganho:>5,} ({ganho_pct:.1f}%)")

df_comp = pd.DataFrame(rows_comp)
df_comp.to_csv(OUT_DIR/"comparacao_estruturado_vs_enriquecido.csv", index=False)

# Gráfico de comparação
fig_cp, ax_cp = plt.subplots(figsize=(10, 5))
x = np.arange(len(rows_comp))
w = 0.35
ax_cp.bar(x - w/2, [r["Estruturado (%)"] for r in rows_comp], w, label="Estruturado (Oracle)", color="steelblue", edgecolor="white")
ax_cp.bar(x + w/2, [r["Enriquecido (%)"] for r in rows_comp], w, label="Enriquecido (texto)", color="seagreen", edgecolor="white")
ax_cp.set_xticks(x); ax_cp.set_xticklabels([r["Campo"] for r in rows_comp])
ax_cp.set_ylabel("Cobertura (%)"); ax_cp.set_title("Ganho de Cobertura: Estruturado → Enriquecido via Texto")
ax_cp.legend(fontsize=10); ax_cp.grid(axis="y", alpha=0.3)
for i, r in enumerate(rows_comp):
    ax_cp.text(i + w/2, r["Enriquecido (%)"] + 0.5, f"+{r['Ganho (%)']:.0f}pp",
               ha="center", va="bottom", fontsize=8, color="darkgreen", fontweight="bold")
plt.tight_layout(); plt.savefig(PLOT_DIR/"08_comparacao_estruturado_enriquecido.png", dpi=150); plt.show(); plt.close()
print("  Gerado: plots/08_comparacao_estruturado_enriquecido.png")

## 10. Triagem Manchester — Distribuição e Associação com Desfechos

In [ ]:
if "cd_cor_manchester" in df.columns and df["cd_cor_manchester"].notna().sum() > 100:
    print("\n" + "="*70)
    print("TRIAGEM MANCHESTER")
    print("="*70)

    cor_map = {1:"Vermelho (imediato)", 2:"Laranja (mt. urgente)", 3:"Amarelo (urgente)",
               4:"Verde (pouco urgente)", 5:"Azul (não urgente)"}
    df["cor_manchester_lbl"] = df["cd_cor_manchester"].map(cor_map).fillna("Desconhecida")
    cores_man = {1:"#d62728", 2:"#ff7f0e", 3:"#ffd700", 4:"#2ca02c", 5:"#1f77b4"}

    manche = df.groupby("cd_cor_manchester").agg(
        n=("cd_atendimento","count"),
        pct_retorno_30d=("y_retorno_30d","mean"),
        pct_retorno_90d=("y_retorno_90d","mean"),
        pct_obito=("y_obito","mean"),
        ig_med=("semanas_gest","median"),
    ).reset_index()
    for c in ["pct_retorno_30d","pct_retorno_90d","pct_obito"]:
        manche[c] = (manche[c]*100).round(2)
    manche["cor_lbl"] = manche["cd_cor_manchester"].map(cor_map).fillna("?")
    print(manche[["cor_lbl","n","pct_retorno_30d","pct_retorno_90d","pct_obito"]].to_string(index=False))
    manche.to_csv(OUT_DIR/"triagem_manchester.csv", index=False)

    fig_m, axes_m = plt.subplots(1, 3, figsize=(18, 5))
    cores_b = [cores_man.get(int(c),  "gray") for c in manche["cd_cor_manchester"]]
    axes_m[0].bar(manche["cor_lbl"], manche["n"], color=cores_b, edgecolor="white", alpha=0.85)
    axes_m[0].set_xticklabels(manche["cor_lbl"], rotation=20, ha="right", fontsize=8)
    axes_m[0].set_title(f"Volume por Cor Manchester\n(n={df['cd_cor_manchester'].notna().sum():,} com triagem)")
    axes_m[0].grid(axis="y", alpha=0.3)

    axes_m[1].bar(manche["cor_lbl"], manche["pct_retorno_30d"], color=cores_b, edgecolor="white", alpha=0.85)
    axes_m[1].axhline(df["y_retorno_30d"].mean()*100, color="black", ls="--", lw=1, label="Média global")
    axes_m[1].set_xticklabels(manche["cor_lbl"], rotation=20, ha="right", fontsize=8)
    axes_m[1].set_title("Retorno 30d por Cor Manchester (%)"); axes_m[1].legend(fontsize=8); axes_m[1].grid(axis="y", alpha=0.3)

    axes_m[2].bar(manche["cor_lbl"], manche["pct_retorno_90d"], color=cores_b, edgecolor="white", alpha=0.85)
    axes_m[2].axhline(df["y_retorno_90d"].mean()*100, color="black", ls="--", lw=1, label="Média global")
    axes_m[2].set_xticklabels(manche["cor_lbl"], rotation=20, ha="right", fontsize=8)
    axes_m[2].set_title("Retorno 90d por Cor Manchester (%)"); axes_m[2].legend(fontsize=8); axes_m[2].grid(axis="y", alpha=0.3)

    plt.suptitle("Triagem Manchester — Distribuição e Desfechos", fontsize=12, y=1.01)
    plt.tight_layout(); plt.savefig(PLOT_DIR/"09_triagem_manchester.png", dpi=150, bbox_inches="tight"); plt.show(); plt.close()
    print("  Gerado: plots/09_triagem_manchester.png")
else:
    print("  Triagem Manchester: dados insuficientes (cd_cor_manchester)")

## 11. Pacientes Multivisita — Padrão de Retorno

In [ ]:
print("\n" + "="*70)
print("ANÁLISE DE PACIENTES MULTIVISITA")
print("="*70)

visitas = df.groupby("cd_paciente").agg(
    n_visitas=("cd_atendimento","count"),
    n_retorno_30d=("y_retorno_30d","sum"),
    primeira_visita=("dt_atendimento","min"),
    ultima_visita=("dt_atendimento","max"),
).reset_index()
visitas["span_dias"] = (visitas["ultima_visita"] - visitas["primeira_visita"]).dt.days
visitas["faixa_visitas"] = pd.cut(visitas["n_visitas"],
    bins=[0,1,2,3,5,10,999], labels=["1","2","3","4-5","6-10",">10"], right=True)

print(f"  Pacientes únicos: {len(visitas):,}")
print(f"  1 visita:  {(visitas['n_visitas']==1).sum():,} ({(visitas['n_visitas']==1).mean()*100:.1f}%)")
print(f"  2 visitas: {(visitas['n_visitas']==2).sum():,} ({(visitas['n_visitas']==2).mean()*100:.1f}%)")
print(f"  ≥5 visitas:{(visitas['n_visitas']>=5).sum():,} ({(visitas['n_visitas']>=5).mean()*100:.1f}%)")
print(f"  Max visitas: {visitas['n_visitas'].max()}")
print(f"  Span mediano (pacientes ≥2 visitas): "
      f"{visitas[visitas['n_visitas']>=2]['span_dias'].median():.0f} dias")

fig_mv, axes_mv = plt.subplots(1, 3, figsize=(18, 5))

vc_vis = visitas["faixa_visitas"].value_counts().sort_index()
axes_mv[0].bar(vc_vis.index.astype(str), vc_vis.values, color="steelblue", edgecolor="white", alpha=0.85)
axes_mv[0].set_xlabel("Nº de visitas"); axes_mv[0].set_ylabel("Nº de pacientes")
axes_mv[0].set_title("Distribuição de Visitas por Paciente"); axes_mv[0].grid(axis="y", alpha=0.3)
for i, v in enumerate(vc_vis.values):
    axes_mv[0].text(i, v + 10, f"{v:,}", ha="center", fontsize=7)

# Retorno 30d por faixa de visitas
ret_vis = df.merge(visitas[["cd_paciente","faixa_visitas"]], on="cd_paciente")
ret_faixa = ret_vis.groupby("faixa_visitas", observed=True)["y_retorno_30d"].mean() * 100
axes_mv[1].bar(ret_faixa.index.astype(str), ret_faixa.values, color="tomato", edgecolor="white", alpha=0.85)
axes_mv[1].axhline(df["y_retorno_30d"].mean()*100, color="black", ls="--", lw=1, label="Média global")
axes_mv[1].set_xlabel("Nº de visitas"); axes_mv[1].set_ylabel("Retorno 30d (%)")
axes_mv[1].set_title("Retorno 30d por Padrão de Visitas"); axes_mv[1].legend(fontsize=8); axes_mv[1].grid(axis="y", alpha=0.3)

# Span dias de acompanhamento (pacientes ≥2 visitas)
visitas[visitas["n_visitas"]>=2]["span_dias"].clip(0, 2000).hist(
    ax=axes_mv[2], bins=40, color="seagreen", edgecolor="white")
axes_mv[2].set_xlabel("Dias de acompanhamento"); axes_mv[2].set_ylabel("N pacientes")
axes_mv[2].set_title("Duração do Acompanhamento\n(pacientes com ≥2 visitas)"); axes_mv[2].grid(alpha=0.3)

plt.suptitle("Padrão de Retorno — Análise de Pacientes Multivisita", fontsize=12, y=1.01)
plt.tight_layout(); plt.savefig(PLOT_DIR/"10_multivisita.png", dpi=150, bbox_inches="tight"); plt.show(); plt.close()
visitas.to_csv(OUT_DIR/"pacientes_multivisita.csv", index=False)
print("  Gerado: plots/10_multivisita.png")

## 12. Análise Univariada — Preditores de Retorno 30d

Risk Ratio (RR) com IC 95% para variáveis categóricas/binárias.
Mann-Whitney U para variáveis contínuas.

In [ ]:
print("\n" + "="*70)
print("ANÁLISE UNIVARIADA — PREDITORES DE RETORNO 30d")
print("="*70)

def rr_ci(cond, outcome, alpha=0.05):
    """Risk Ratio com IC 95% (método log)."""
    a = int((cond & (outcome==1)).sum())
    b = int((cond & (outcome==0)).sum())
    c = int((~cond & (outcome==1)).sum())
    d = int((~cond & (outcome==0)).sum())
    n1, n0 = a + b, c + d
    if n1 == 0 or n0 == 0 or a == 0 or c == 0:
        return np.nan, np.nan, np.nan, np.nan
    p1 = a / n1; p0 = c / n0
    rr = p1 / p0
    se = np.sqrt(1/a - 1/n1 + 1/c - 1/n0)
    z  = scipy_stats.norm.ppf(1 - alpha/2)
    lo, hi = np.exp(np.log(rr) - z*se), np.exp(np.log(rr) + z*se)
    # Chi² p-valor
    try:
        ct = np.array([[a,b],[c,d]])
        _, p, _, _ = scipy_stats.chi2_contingency(ct)
    except Exception:
        p = np.nan
    return round(rr,3), round(lo,3), round(hi,3), round(p,5)

rows_uv = []

# Variáveis binárias / categóricas
preds_bin = {
    "tp_atend == U (urgência)": df["tp_atendimento"] == "U",
    "tp_atend == I (internação)": df["tp_atendimento"] == "I",
    "Idade < 10a (caso extremo)":     df["idade_anos"] < 10,
    "Idade 10–14a (abuso/prec.)":   df["idade_anos"].between(10, 14),
    "Idade 15–19a (adolescente)":   df["idade_anos"].between(15, 19),
    "Idade ≥ 35a (id. avançada)": df["idade_anos"] >= 35,
    "Prematuro (<37s)": df["semanas_gest"] < 37,
    "Muito prematuro (<32s)": df["semanas_gest"] < 32,
    "G_obs ≥ 3 (multípara)": df["G_obs"] >= 3 if "G_obs" in df.columns else pd.Series(False, index=df.index),
    "A_obs ≥ 1 (aborto prévio)": df["A_obs"] >= 1 if "A_obs" in df.columns else pd.Series(False, index=df.index),
    "A_obs ≥ 2 (aborto recorrente)": df["A_obs"] >= 2 if "A_obs" in df.columns else pd.Series(False, index=df.index),
    "C_obs ≥ 1 (cesárea prévia)": df["C_obs"] >= 1 if "C_obs" in df.columns else pd.Series(False, index=df.index),
    "Sem pré-natal": (pd.to_numeric(df.get("nr_consultas_pre_natal"), errors="coerce").fillna(-1)==0),
    "Manchester vermelho/laranja": df.get("cd_cor_manchester", pd.Series()).fillna(5) <= 2 if "cd_cor_manchester" in df.columns else pd.Series(False, index=df.index),
    "Hipertensivos (CID)": df["segmento"] == "Hipertensivos",
    "Diabetes Gestacional (CID)": df["segmento"] == "Diabetes Gest.",
    "Parto Prematuro (CID)": df["segmento"] == "Parto Prematuro",
    "Retorno anterior": df.get("cd_atendimento_pai", pd.Series()).notna() if "cd_atendimento_pai" in df.columns else pd.Series(False, index=df.index),
}

y = df["y_retorno_30d"]
for lbl, cond in preds_bin.items():
    cond = cond.fillna(False).astype(bool)
    n_exp = int(cond.sum())
    if n_exp < 10:
        continue
    rr, lo, hi, p = rr_ci(cond, y)
    rows_uv.append({"Preditor": lbl, "N exposto": n_exp,
                    "Retorno(exp)%": round(y[cond].mean()*100, 1),
                    "Retorno(nexp)%": round(y[~cond].mean()*100, 1),
                    "RR": rr, "IC95_lo": lo, "IC95_hi": hi, "p_valor": p})

tab_uv = pd.DataFrame(rows_uv).sort_values("RR", ascending=False)
print(tab_uv.to_string(index=False))
tab_uv.to_csv(OUT_DIR/"analise_univariada_retorno30d.csv", index=False)

# Forest plot de RR
tab_plot = tab_uv.dropna(subset=["RR"]).sort_values("RR")
fig_uv, ax_uv = plt.subplots(figsize=(10, max(6, len(tab_plot)*0.45)))
y_pos = range(len(tab_plot))
ax_uv.errorbar(
    tab_plot["RR"], y_pos,
    xerr=[tab_plot["RR"] - tab_plot["IC95_lo"], tab_plot["IC95_hi"] - tab_plot["RR"]],
    fmt="o", color="steelblue", capsize=4, markersize=5, elinewidth=1.2,
)
ax_uv.axvline(1.0, color="black", ls="--", lw=0.8)
ax_uv.set_yticks(list(y_pos)); ax_uv.set_yticklabels(tab_plot["Preditor"], fontsize=9)
ax_uv.set_xlabel("Risk Ratio (IC 95%)")
ax_uv.set_title("Forest Plot — Preditores Univariados de Retorno 30d\n(RR > 1 = maior risco de retorno)")
ax_uv.grid(axis="x", alpha=0.3)
for i, (_, row) in enumerate(tab_plot.iterrows()):
    p_str = f"p={row['p_valor']:.4f}" if not np.isnan(row["p_valor"]) else ""
    ax_uv.text(tab_plot["IC95_hi"].max() + 0.05, i,
               f"RR={row['RR']:.2f}  {p_str}", va="center", fontsize=7)
plt.tight_layout(); plt.savefig(PLOT_DIR/"11_forest_plot_univariado.png", dpi=150, bbox_inches="tight"); plt.show(); plt.close()
print("  Gerado: plots/11_forest_plot_univariado.png")

## 13. Matriz de Correlação — Features Numéricas Principais

In [ ]:
feats_corr = [f for f in ["idade_anos","semanas_gest","G_obs","A_obs","C_obs","IG_obs",
                            "vl_escore_triagem","cd_cor_manchester","qtd_problemas",
                            "qtd_evolucoes_documentadas","qtd_exames_solicitados",
                            "duracao_horas_aprox","y_retorno_30d","y_retorno_90d"]
               if f in df.columns and pd.api.types.is_numeric_dtype(df[f])]

df_corr = df[feats_corr].apply(pd.to_numeric, errors="coerce")
corr_mat = df_corr.corr(method="spearman")  # Spearman para dados não-normais

fig_cr, ax_cr = plt.subplots(figsize=(12, 10))
mask = np.triu(np.ones_like(corr_mat, dtype=bool), k=1)
sns.heatmap(corr_mat, ax=ax_cr, annot=True, fmt=".2f", cmap="RdBu_r",
            center=0, vmin=-1, vmax=1, linewidths=0.5,
            annot_kws={"size": 8}, mask=mask)
ax_cr.set_title("Matriz de Correlação de Spearman — Features Numéricas\n"
                "(destaque: correlações com y_retorno_30d)", fontsize=12)
plt.tight_layout(); plt.savefig(PLOT_DIR/"12_correlacao_spearman.png", dpi=150, bbox_inches="tight"); plt.show(); plt.close()
corr_mat.to_csv(OUT_DIR/"correlacao_spearman.csv")
print("  Gerado: plots/12_correlacao_spearman.png")

## Resumo Final

In [ ]:
n_plots = len(list(PLOT_DIR.iterdir()))
print(f"""
╔══════════════════════════════════════════════════════════════════╗
║  EDA — Gestante de Alto Risco — CONCLUÍDO                        ║
╠══════════════════════════════════════════════════════════════════╣
║  Coorte final: {n_final:<10,}  Pacientes únicos: {df['cd_paciente'].nunique():<8,}  ║
║  Período: {df['dt_atendimento'].min().date()} → {df['dt_atendimento'].max().date()}       ║
║  Retorno 30d: {df['y_retorno_30d'].mean()*100:.1f}%  |  Retorno 90d: {df['y_retorno_90d'].mean()*100:.1f}%     ║
╠══════════════════════════════════════════════════════════════════╣
║  CSVs gerados:                                                   ║
║   tabela1_coorte.csv              (Table 1 com p-valores)        ║
║   qualidade_campos.csv            (cobertura + enriquecidos)     ║
║   perfil_segmento.csv             (por CID)                      ║
║   comparacao_estruturado_enriquecido.csv  (ganho texto)          ║
║   triagem_manchester.csv          (Manchester × desfechos)       ║
║   pacientes_multivisita.csv       (padrão de retorno)            ║
║   analise_univariada_retorno30d.csv (RR + IC95% + p)            ║
║   correlacao_spearman.csv         (matriz de correlação)         ║
║   eda_recem_nascidos.csv          (RN vinculados)                ║
╠══════════════════════════════════════════════════════════════════╣
║  Plots gerados: {n_plots:<3} figuras                                    ║
║   00  CONSORT flow                                               ║
║   01  Cobertura campos-chave                                     ║
║   02  Distribuição temporal + sazonalidade                       ║
║   03  Perfil por segmento CID                                    ║
║   04  Demográfico + obstétrico                                   ║
║   05  Risco LLM                                                  ║
║   06  EDA Recém-Nascidos                                         ║
║   07  Perfil obstétrico enriquecido (G/A/IG/tipo)               ║
║   08  Estruturado vs Enriquecido (ganho de cobertura)            ║
║   09  Triagem Manchester × desfechos                             ║
║   10  Multivisita — padrão de retorno                            ║
║   11  Forest plot univariado (RR retorno 30d)                    ║
║   12  Matriz de correlação Spearman                              ║
╚══════════════════════════════════════════════════════════════════╝
""")